In [11]:
# Import necessary libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

import swcol as sw

In [12]:
scenario_path = '../../data/Colombia/scenarios/1/'
model_inputs_path = scenario_path+'inputs/'
model_outputs_path = scenario_path+'outputs/'
gen_per_res_vs_marg_cost = '../../data/XM-API/Plan/gen_per_res_vs_marg_cost/Esc1.csv'
emissions = '../../data/XM-API/Plan/emissions/esc1.csv'
cap_2023 = '../../data/XM-API/Plan/installed_capacity/esc1/2023.csv'
cap_2037 = '../../data/XM-API/Plan/installed_capacity/esc1/2037.csv'
dema_path = '../../data/XM-API/variable_query/2022-12-01_2023-11-30/'

In [13]:
sw.scenarios.table(model_outputs_path, model_inputs_path)

Year,2023,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037
Tech,,,,,,,,,,,,,,,
Biomass,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.00,56.87,38.00,0.00,5.12,0.00,0.0,0.0
Eolica,20.00,1122.08,0.0,0.0,255.0,450.0,492.0,0.00,75.33,415.02,488.94,126.42,463.14,0.0,0.0
Geothermal,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.00,57.89,60.44,0.00,31.66,0.00,0.0,0.0
Hidro,45.64,0.00,0.0,1200.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0
Thermal,0.00,52.00,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0
pv_solar,466.60,1232.73,110.0,218.0,128.0,128.0,123.0,256.87,114.33,91.00,79.00,80.00,1016.97,51.0,69.0
Total,532.24,2406.81,110.0,1418.0,383.0,578.0,615.0,256.87,304.42,604.46,567.94,243.20,1480.11,51.0,69.0


In [14]:
# Transform data
sce_sw = pd.read_csv(model_outputs_path+'/dispatch.csv')
sce_sw['year'] = sce_sw['timestamp'].str[:4]
sce_sw['Energy_TWh_typical_yr'] = sce_sw['Energy_GWh_typical_yr'] / 1000
sce_sw.rename(columns={'gen_tech': 'Tech'}, inplace=True)
sce_sw = sce_sw[sce_sw['timestamp'].str.contains('^2023') == False]
# Plot
sw.scenarios.dispatched_generation(sce_sw, 'year', 'Energy_TWh_typical_yr', 'Tech')

In [15]:
sce_upme = pd.read_csv(gen_per_res_vs_marg_cost)
sce_upme['Energy_TWh_typical_yr'] = sce_upme['Energy_GWh_typical_yr'] / 1000
sw.scenarios.dispatched_generation(sce_upme, 'year', 'Energy_TWh_typical_yr', 'Tech')

In [16]:
em_sw = pd.read_csv(model_outputs_path+'/emissions.csv')
em_sw['AnnualEmissions_MtCO2_per_yr'] = em_sw['AnnualEmissions_tCO2_per_yr'] / 1000000
em_sw = em_sw[em_sw['PERIOD'] != 2023]
sw.scenarios.annual_emmissions(em_sw, 'PERIOD', 'AnnualEmissions_MtCO2_per_yr')

In [17]:
em_upme = pd.read_csv(emissions)
em_sw = pd.read_csv(model_outputs_path+'/emissions.csv')
sw.scenarios.annual_emmissions(em_upme, 'PERIOD', 'AnnualEmissions_tCO2_per_yr')

In [18]:
cap_inst = pd.read_csv(model_outputs_path+'BuildGen.csv')
gen_info = pd.read_csv(model_inputs_path+'gen_info.csv')
cap_inst = pd.merge(cap_inst, gen_info, left_on='GEN_BLD_YRS_1',
    right_on='GENERATION_PROJECT', how='inner')
## Remove unactive plants
cap_inst = cap_inst[cap_inst['gen_max_age'] > 1]

cap_inst = cap_inst[['GENERATION_PROJECT','gen_tech','GEN_BLD_YRS_2','BuildGen']]
cap_inst_2023 = cap_inst[cap_inst['GEN_BLD_YRS_2'] <= 2023].copy()
cap_inst_2023.rename(columns={'gen_tech': 'Tech'}, inplace=True)
cap_inst_2023 = cap_inst_2023.groupby(['Tech']).agg({
    'BuildGen': 'sum'
}).reset_index()
cap_inst_2023['BuildGen'] = 100 * cap_inst_2023['BuildGen'] / cap_inst_2023['BuildGen'].sum()

cap_inst = cap_inst[cap_inst['GEN_BLD_YRS_2'] <= 2037].copy()
cap_inst.rename(columns={'gen_tech': 'Tech'}, inplace=True)

sw.scenarios.installed_capacity(cap_inst_2023, cap_inst, "Switch")

In [19]:
import pandas as pd
esc_cap_2023 = pd.read_csv(cap_2023)
esc_cap_2023.rename(columns={'gen_tech': 'Tech'}, inplace=True)
esc_cap_2037 = pd.read_csv(cap_2037)
esc_cap_2037.rename(columns={'gen_tech': 'Tech'}, inplace=True)

sw.scenarios.installed_capacity(esc_cap_2023, esc_cap_2037, "UPME")